# E2VID — Training Notebook

**Hybrid Vision · AMI 2026**

Train (or fine-tune) the **E2VID** recurrent UNet to reconstruct intensity frames from event-camera voxel grids.

| Setting | Value |
|---|---|
| Model | Recurrent UNet (Rebecq et al., CVPR 2019) |
| Dataset | Raw `.txt` event files (`t x y p`) |
| Input | Voxel grid `(num_bins, H, W)` |
| Target | Event-count frame `(1, H, W)` in [0, 1] — self-supervised proxy |
| Loss | λ·L1 + λ·(1−SSIM) + λ·TV |
| Optim | AdamW + Warmup-Cosine LR |

---

**Workflow**
1. Install dependencies
2. Clone repo & configure paths
3. Build dataset
4. Define loss & training loop
5. Run training
6. Visualise reconstructions

## 1 — Install dependencies

In [ ]:
# Run only if packages are not yet installed
# !pip install -q torch torchvision gdown h5py opencv-python-headless pyyaml

## 2 — Clone repo & set up paths

In [ ]:
import os, sys

# ── Google Colab ──────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
REPO = "/content/AMI-KK" if IN_COLAB else os.path.abspath("..")

if IN_COLAB and not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/Dark-Fantasy-K/AMI-KK.git {REPO}

if REPO not in sys.path:
    sys.path.insert(0, REPO)

os.chdir(REPO)
print("Working directory:", os.getcwd())

## 3 — Configuration

Edit the paths and hyperparameters here.

In [ ]:
import sys
from pathlib import Path
import torch

# ── Runtime detection ─────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

# ── Data source ───────────────────────────────────────────────────────────
DATA_DIR        = Path("data")   # folder containing *.txt event files
WINDOW_DURATION = 0.05           # seconds per event window (50 ms)
SEQ_LEN         = 10             # consecutive windows per training item
VAL_FRACTION    = 0.1            # fraction of sequences reserved for val
HEIGHT          = None           # sensor height in px; None = auto-detect
WIDTH           = None           # sensor width  in px; None = auto-detect

# ── RAM precompute ────────────────────────────────────────────────────────
# All voxels & targets are computed once and held in RAM.
# Memory at float32: N_windows × (NUM_BINS+1) × H_out × W_out × 4 bytes
#   INPUT_SIZE=256  → ~3.4 GB for 2155 windows  ✓ recommended
#   INPUT_SIZE=None → ~40  GB at 720×1280        ✗ too large for Colab
INPUT_SIZE = 256

WEIGHTS  = None
# WEIGHTS = "models/reconstruction/e2vid/pretrained_weights/E2VID_lightweight.pth.tar"
SAVE_DIR = Path("checkpoints/e2vid")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Voxel ─────────────────────────────────────────────────────────────────
NUM_BINS = 5

# ── DataLoader workers ────────────────────────────────────────────────────
# Colab: num_workers > 0 forces large tensors through multiprocessing IPC
# (131 MB per sample × 2 = 262 MB/batch serialised through Queue each step).
# Setting 0 keeps everything in the main process and is faster in Colab.
# On a local multi-core machine you can raise this to 4.
NUM_WORKERS = 0 if IN_COLAB else 4

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 4         # T4 has 16 GB VRAM — increase from 2 for better GPU utilisation
LR           = 1e-4
WEIGHT_DECAY = 1e-5
WARMUP       = 3
TBPTT_STEPS  = 5
GRAD_CLIP    = 1.0
USE_AMP      = True      # mixed precision — essential on T4

# ── Loss weights ──────────────────────────────────────────────────────────
LAMBDA_L1   = 1.0
LAMBDA_SSIM = 0.5
LAMBDA_TV   = 1e-4

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Runtime : {'Colab' if IN_COLAB else 'local'}")
print(f"Device  : {DEVICE}  |  NUM_WORKERS={NUM_WORKERS}  |  BATCH_SIZE={BATCH_SIZE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4 — Dataset

**Data format** — each `.txt` file: one event per line, `t x y p` (p ∈ {−1, +1}).

**Pipeline (runs once before training):**

```
txt files → load raw events into RAM
          → searchsorted → window indices (fi, i0, i1)
          → precompute_windows → voxel + target arrays in RAM
          → group into SEQ_LEN sequences → PrecomputedEventDataset
```

After precomputation, `__getitem__` is a single numpy fancy-index — no disk I/O, no CPU compute during training.

**Target** — event-count frame (events-per-pixel, normalised to [0,1]). Self-supervised proxy; replace with real GT frames if available.

In [ ]:
from __future__ import annotations
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# ── Event file loader ──────────────────────────────────────────────────────

def _load_txt_events(path: Path) -> dict:
    """
    Load 't x y p' event file.  p ∈ {-1, +1}.

    First call: parses the .txt and writes a .npy cache next to it.
    Later calls: loads the .npy directly (~10× faster than text parsing).
    """
    cache = path.with_suffix(".npy")
    if cache.exists():
        data = np.load(cache)           # float32 (N, 4), binary read
    else:
        print(f"    Parsing {path.name} and caching to .npy …")
        raw  = np.fromfile(str(path), dtype=np.float32, sep=" ")
        data = raw[: len(raw) // 4 * 4].reshape(-1, 4)
        np.save(cache, data)            # saves ~half the size of the txt
        print(f"    Cached → {cache.name}")

    return {
        "t": data[:, 0].astype(np.float64),
        "x": data[:, 1].astype(np.int32),
        "y": data[:, 2].astype(np.int32),
        "p": data[:, 3],    # float32, already {-1.0, +1.0}
    }


# ── Voxel grid builder (bincount — 10-50× faster than np.add.at) ──────────

def _events_to_voxel(x, y, t, p, H, W, num_bins, t_start, t_end):
    """
    Build (num_bins, H, W) voxel grid.  p must be float32 in {-1, +1}.

    Uses np.bincount instead of np.add.at:
      - np.add.at  : unbuffered scatter, no SIMD  → ~5-15 ms / call
      - np.bincount: fully vectorised C loop       → ~0.1-0.5 ms / call
    """
    if len(x) == 0 or t_end <= t_start:
        return np.zeros((num_bins, H, W), dtype=np.float32)

    t_norm = (t - t_start) / (t_end - t_start) * (num_bins - 1)
    tb     = np.clip(t_norm.astype(np.int32), 0, num_bins - 2)
    alpha  = (t_norm - tb).astype(np.float32)

    valid = (x >= 0) & (x < W) & (y >= 0) & (y < H)
    xv = x[valid].astype(np.int64)
    yv = y[valid].astype(np.int64)
    tbv, av, pv = tb[valid].astype(np.int64), alpha[valid], p[valid]

    HW   = H * W
    flat = tbv * HW + yv * W + xv          # flat index into (num_bins, H, W)

    # Two bincount calls replace two np.add.at calls
    vox = (
        np.bincount(flat,      weights=(1.0 - av) * pv, minlength=num_bins * HW) +
        np.bincount(flat + HW, weights=av * pv,          minlength=num_bins * HW)
    )
    voxel = vox.reshape(num_bins, H, W).astype(np.float32)

    for b in range(num_bins):
        m = np.abs(voxel[b]).max()
        if m > 0:
            voxel[b] /= m

    return voxel


# ── One-shot precomputation ────────────────────────────────────────────────

def precompute_windows(events_list, all_wins, H, W, num_bins, input_size=None):
    """
    Compute voxel grids and event-count targets for every window in all_wins.
    Each unique window is computed once; results live in two contiguous RAM arrays.

    Returns:
        voxels  : float32  (N, num_bins, H_out, W_out)
        targets : float32  (N, 1,        H_out, W_out)
    """
    H_out = input_size or H
    W_out = input_size or W
    N     = len(all_wins)
    mem_gb = N * (num_bins + 1) * H_out * W_out * 4 / 1e9
    print(f"Allocating: {N} windows × ({num_bins}+1) × {H_out}×{W_out}  =  {mem_gb:.2f} GB")

    voxels  = np.zeros((N, num_bins, H_out, W_out), dtype=np.float32)
    targets = np.zeros((N, 1,        H_out, W_out), dtype=np.float32)

    for k, (fi, i0, i1) in enumerate(all_wins):
        if k % 200 == 0:
            print(f"  {k:>5}/{N}", end="\r", flush=True)

        ev = events_list[fi]
        x, y = ev["x"][i0:i1], ev["y"][i0:i1]
        t, p = ev["t"][i0:i1], ev["p"][i0:i1]

        t0, t1 = float(t[0]), float(t[-1])
        vox = _events_to_voxel(x, y, t, p, H, W, num_bins, t0, t1)

        # Target: event-count frame via bincount (also avoids np.add.at)
        valid    = (x >= 0) & (x < W) & (y >= 0) & (y < H)
        flat_tgt = y[valid].astype(np.int64) * W + x[valid].astype(np.int64)
        tgt      = np.bincount(flat_tgt, minlength=H * W).reshape(H, W).astype(np.float32)
        m = tgt.max()
        if m > 0:
            tgt /= m

        if input_size is not None:
            vox = np.stack([
                cv2.resize(vox[b], (W_out, H_out), interpolation=cv2.INTER_LINEAR)
                for b in range(num_bins)
            ])
            tgt = cv2.resize(tgt, (W_out, H_out), interpolation=cv2.INTER_LINEAR)

        voxels[k]     = vox
        targets[k, 0] = tgt

    print(f"  {N}/{N}  done")
    print(f"Voxels : {voxels.nbytes  / 1e9:.2f} GB in RAM")
    print(f"Targets: {targets.nbytes / 1e9:.2f} GB in RAM")
    return voxels, targets


# ── Dataset — pure RAM indexer ─────────────────────────────────────────────

class PrecomputedEventDataset(Dataset):
    """
    Indexes pre-computed voxel and target arrays by sequence.
    __getitem__ = one numpy fancy-index + optional horizontal flip. Zero compute.
    """

    def __init__(self, voxels, targets, seqs, augment=False):
        self.voxels  = voxels
        self.targets = targets
        self.seqs    = [np.asarray(s, dtype=np.int64) for s in seqs]
        self.augment = augment

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids = self.seqs[idx]
        v   = self.voxels[ids]            # (SEQ_LEN, num_bins, H, W)
        t   = self.targets[ids]           # (SEQ_LEN, 1, H, W)
        if self.augment and random.random() < 0.5:
            v = v[..., ::-1].copy()
            t = t[..., ::-1].copy()
        return torch.from_numpy(v), torch.from_numpy(t)

In [ ]:
from collections import defaultdict

# ── 1. Discover txt files ─────────────────────────────────────────────────
txt_files = sorted(DATA_DIR.glob("**/*.txt"))
assert txt_files, f"No .txt files found under {DATA_DIR}"
print(f"Found {len(txt_files)} file(s): {[f.name for f in txt_files]}")

# ── 2. Auto-detect sensor resolution ─────────────────────────────────────
_H, _W = HEIGHT, WIDTH
if _H is None or _W is None:
    _raw  = np.fromfile(str(txt_files[0]), dtype=np.float32, sep=" ")
    _raw  = _raw[: len(_raw) // 4 * 4].reshape(-1, 4)
    _H, _W = int(_raw[:, 2].max()) + 1, int(_raw[:, 1].max()) + 1
    del _raw
print(f"Sensor: H={_H}, W={_W}")

# ── 3. Load raw events into RAM (once) ────────────────────────────────────
print("\nLoading events into RAM …")
events_list = []
for f in txt_files:
    ev  = _load_txt_events(f)
    dur = ev["t"][-1] - ev["t"][0]
    print(f"  {f.name}: {len(ev['t']):,} events  {dur:.1f} s")
    events_list.append(ev)

# ── 4. Index time windows (searchsorted — O(log N) per window) ────────────
print("\nIndexing time windows …")
all_wins = []    # [(file_idx, i_start, i_end), ...]
for fi, ev in enumerate(events_list):
    t = ev["t"]
    for ws in np.arange(t[0], t[-1], WINDOW_DURATION):
        i0 = int(np.searchsorted(t, ws,                    "left"))
        i1 = int(np.searchsorted(t, ws + WINDOW_DURATION,  "left"))
        if i1 - i0 >= 50:
            all_wins.append((fi, i0, i1))
print(f"  {len(all_wins)} windows")

# ── 5. Group windows into sequences ───────────────────────────────────────
wins_by_file = defaultdict(list)
for k, (fi, _, _) in enumerate(all_wins):
    wins_by_file[fi].append(k)

stride   = max(1, SEQ_LEN // 2)
all_seqs = []
for fi in sorted(wins_by_file):
    ids = wins_by_file[fi]
    for s in range(0, len(ids) - SEQ_LEN + 1, stride):
        all_seqs.append(ids[s : s + SEQ_LEN])

cut = max(1, int(len(all_seqs) * (1 - VAL_FRACTION)))
train_seqs, val_seqs = all_seqs[:cut], all_seqs[cut:]
print(f"  {len(all_seqs)} sequences  →  {len(train_seqs)} train / {len(val_seqs)} val")

# ── 6. Precompute ALL voxels & targets into RAM ───────────────────────────
# Each unique window is computed exactly once; train and val share the arrays.
print()
voxels, targets = precompute_windows(
    events_list, all_wins, _H, _W, NUM_BINS, input_size=INPUT_SIZE
)

# ── 7. Build datasets (zero-copy views into the same RAM arrays) ──────────
train_ds = PrecomputedEventDataset(voxels, targets, train_seqs, augment=True)
val_ds   = PrecomputedEventDataset(voxels, targets, val_seqs,   augment=False)

# On Linux, forked workers share the numpy arrays via copy-on-write → safe.
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    drop_last=True, persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
print(f"\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# Sanity check
v, t = train_ds[0]
print(f"Voxel  shape: {v.shape}  dtype={v.dtype}  range=[{v.min():.2f}, {v.max():.2f}]")
print(f"Target shape: {t.shape}  dtype={t.dtype}  range=[{t.min():.2f}, {t.max():.2f}]")

## 5 — Model

In [ ]:
from models.reconstruction.e2vid.model import E2VID, load_e2vid

if WEIGHTS:
    model = load_e2vid(WEIGHTS, DEVICE)
    model.train()
    print(f"Fine-tuning from: {WEIGHTS}")
else:
    model = E2VID(num_bins=NUM_BINS).to(DEVICE)
    print("Training from scratch")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  (~{n_params/1e6:.1f} M)")

## 6 — Loss functions

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def _ssim(pred, target, k=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    pad = k // 2
    def pool(x):
        return F.avg_pool2d(x, k, stride=1, padding=pad)
    mu_p, mu_t = pool(pred), pool(target)
    sp  = pool(pred   * pred)   - mu_p * mu_p
    st  = pool(target * target) - mu_t * mu_t
    spt = pool(pred   * target) - mu_p * mu_t
    num = (2 * mu_p * mu_t + C1) * (2 * spt + C2)
    den = (mu_p**2 + mu_t**2 + C1) * (sp + st + C2)
    return (num / den).mean()


def _tv(x):
    return (
        (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
        + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    )


class E2VIDLoss(nn.Module):
    def __init__(self, w_l1=1.0, w_ssim=0.5, w_tv=1e-4):
        super().__init__()
        self.w_l1, self.w_ssim, self.w_tv = w_l1, w_ssim, w_tv

    def forward(self, pred, target):
        l1   = F.l1_loss(pred, target)
        ssim = 1.0 - _ssim(pred, target)
        tv   = _tv(pred)
        loss = self.w_l1 * l1 + self.w_ssim * ssim + self.w_tv * tv
        return loss, {"l1": l1.item(), "ssim": ssim.item(), "tv": tv.item()}


criterion = E2VIDLoss(LAMBDA_L1, LAMBDA_SSIM, LAMBDA_TV)
print("Loss: E2VIDLoss(L1 + SSIM + TV) ready")

## 7 — Optimizer & scheduler

In [ ]:
import math

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / max(WARMUP, 1)
    t = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler() if (USE_AMP and DEVICE.type == "cuda") else None

print(f"Optimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}")
print(f"Scheduler : warmup-{WARMUP} + cosine  AMP={'on' if scaler else 'off'}")

## 8 — Training & validation helpers

In [ ]:
def detach_states(model):
    """Stop gradients flowing through LSTM states between TBPTT chunks."""
    for i, s in enumerate(model.unet.states):
        if s is not None:
            h, c = s
            model.unet.states[i] = (h.detach(), c.detach())


def train_one_epoch(model, loader, optimizer, criterion, device,
                    tbptt_steps=5, grad_clip=1.0, scaler=None):
    model.train()
    totals = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0

    for voxels, targets in loader:
        voxels  = voxels.to(device, non_blocking=True)   # (B, K, bins, H, W)
        targets = targets.to(device, non_blocking=True)  # (B, K, 1, H, W)
        K = voxels.shape[1]
        model.unet.reset_states()

        for t0 in range(0, K, tbptt_steps):
            t1 = min(t0 + tbptt_steps, K)
            optimizer.zero_grad()
            chunk_loss = torch.tensor(0.0, device=device)

            for t in range(t0, t1):
                with torch.autocast(device_type=device.type, enabled=(scaler is not None)):
                    pred = model(voxels[:, t])
                    loss, parts = criterion(pred, targets[:, t])
                chunk_loss = chunk_loss + loss

            if scaler:
                scaler.scale(chunk_loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                chunk_loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            detach_states(model)
            totals["loss"] += chunk_loss.item() / (t1 - t0)
            totals["l1"]   += parts["l1"]
            totals["ssim"] += parts["ssim"]
            n += 1

    return {k: v / max(n, 1) for k, v in totals.items()}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    totals = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for voxels, targets in loader:
        voxels  = voxels.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        model.unet.reset_states()
        for t in range(voxels.shape[1]):
            pred = model(voxels[:, t])
            loss, parts = criterion(pred, targets[:, t])
            totals["loss"] += loss.item()
            totals["l1"]   += parts["l1"]
            totals["ssim"] += parts["ssim"]
            n += 1
    return {k: v / max(n, 1) for k, v in totals.items()}

print("Helpers defined.")

## 9 — Run training

Metrics are logged per epoch. Checkpoints are saved to `SAVE_DIR/last.pth` and `best.pth`.

In [ ]:
import time

history = {"train_loss": [], "val_loss": [], "lr": []}
best_val = float("inf")

for epoch in range(EPOCHS):
    t0 = time.time()

    tr = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE,
        tbptt_steps=TBPTT_STEPS, grad_clip=GRAD_CLIP, scaler=scaler,
    )
    vl = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    elapsed = time.time() - t0
    current_lr = scheduler.get_last_lr()[0]

    history["train_loss"].append(tr["loss"])
    history["val_loss"].append(vl["loss"])
    history["lr"].append(current_lr)

    print(
        f"Epoch {epoch+1:3d}/{EPOCHS} | {elapsed:4.0f}s | lr={current_lr:.2e} | "
        f"train: loss={tr['loss']:.4f} L1={tr['l1']:.4f} SSIM={tr['ssim']:.4f} | "
        f"val: loss={vl['loss']:.4f} L1={vl['l1']:.4f}"
    )

    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val,
    }
    torch.save(ckpt, SAVE_DIR / "last.pth")

    if vl["loss"] < best_val:
        best_val = vl["loss"]
        torch.save(ckpt, SAVE_DIR / "best.pth")
        print(f"  ✓ New best val loss: {best_val:.4f}")

print(f"\nTraining complete. Best val loss: {best_val:.4f}")

## 10 — Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("E2VID Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["lr"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("LR Schedule (warmup + cosine)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "loss_curves.png", dpi=120)
plt.show()

## 11 — Visualise reconstructions from validation set

In [ ]:
import matplotlib.pyplot as plt

model.eval()
model.unet.reset_states()

voxels, targets = next(iter(val_loader))
voxels  = voxels.to(DEVICE)
targets = targets.to(DEVICE)

K       = voxels.shape[1]
N_SHOW  = min(6, K)
step    = max(1, K // N_SHOW)
frames  = list(range(0, K, step))[:N_SHOW]

preds = []
with torch.no_grad():
    for t in range(K):
        p = model(voxels[:, t])
        if t in frames:
            preds.append(p[0, 0].cpu().numpy())

fig, axes = plt.subplots(2, N_SHOW, figsize=(3 * N_SHOW, 6))
for i, (fi, pred) in enumerate(zip(frames, preds)):
    gt = targets[0, fi, 0].cpu().numpy()
    axes[0, i].imshow(gt,   cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"GT  frame {fi}")
    axes[0, i].axis("off")
    axes[1, i].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Pred frame {fi}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Ground truth", fontsize=10)
axes[1, 0].set_ylabel("E2VID output", fontsize=10)
plt.suptitle("E2VID Reconstructions (validation)", fontsize=13)
plt.tight_layout()
plt.savefig(SAVE_DIR / "sample_reconstructions.png", dpi=120)
plt.show()

## 12 — Load best checkpoint & export

In [ ]:
# Load the best checkpoint saved during training
best_ckpt_path = SAVE_DIR / "best.pth"
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch'] + 1}")
print(f"Best val loss: {ckpt['best_val_loss']:.4f}")

In [ ]:
# Optional: export just the model weights in rpg_e2vid-compatible format
export_path = SAVE_DIR / "e2vid_trained.pth"
torch.save({"state_dict": model.state_dict()}, export_path)
print(f"Model weights exported to: {export_path}")
print("Use with: load_e2vid(export_path, device)")